# Technical Documentation
## 1.Project Overview
### 1.1 Background
Mussels are commonly used as model organisms for monitoring marine environments, and their tissue pathology indicators can be used to assess the health of marine life. Traditional manual pathological analysis methods are inefficient, costly, and highly subjective. This project uses deep learning technology and the U-Net image segmentation model to achieve automated, high-throughput quantitative analysis of mussel gill tissue sections.
### 1.2 Objectives
Build a deep‑learning image segmentation model based on the U‑Net architecture.

Automatically segment the gill filament epithelium regions from pathological images of mussel tissues.

Output binary segmentation masks to support subsequent quantitative assessment of pathological damage.

### 1.3 Technical Approach
We adopt an encoder‑decoder U‑Net model with ResNet34 as the encoder backbone. The model takes 512×512 pixel pathological images and performs pixel‑wise binary classification to output segmentation masks of the gill filament epithelium and digestive gland lumens..

## 2. Runtime Environment and Dependencies
### 2.1 Hardware Requirements
Item	Recommended Specification
CPU	   Intel Xeon / Core i7 or higher
GPU	   NVIDIA GPU with ≥8 GB VRAM, CUDA‑capable
RAM	   ≥16 GB
Storage	≥50 GB free space
### 2.2 Software Environment
Item	         Version
Operating System   Ubuntu 18.04+ / Windows 10+ / macOS
Python	         3.8+
CUDA	         11.0+ (for GPU training)
cuDNN	         8.0+ (for GPU training)
### 2.3 Python Dependencies
The core dependencies (derived from the provided code) are:


In [ ]:
# Core frameworks
tensorflow>=2.8.0
keras>=2.8.0

# Segmentation model libraries
segmentation_models          # provides U‑Net and other segmentation models
keras-unet-collection        # additional U‑Net variants

# Data processing and scientific computing
numpy>=1.19.0
matplotlib>=3.3.0

# Medical image evaluation metrics
medpy                         # provides Dice, HD95, etc.

# Others
glob                          # file path handling
os                            # operating system interface

### 2.4 Installation Steps

In [ ]:
# 1. Create a virtual environment (recommended)
conda create -n mussel_seg python=3.9
conda activate mussel_seg

# 2. Install TensorFlow (GPU version)
pip install tensorflow-gpu==2.8.0

# 3. Install segmentation model libraries
pip install segmentation-models
pip install keras-unet-collection

# 4. Install other dependencies
pip install numpy matplotlib medpy

## 3. Project Directory Structure
Based on the file screenshot, the project structure is as follows:

In [ ]:
Project Root/
├── keras_unet_collection/          # keras-unet-collection source
├── keras-unet-collection-main/     # main library directory
├── MMHID/                          # dataset directory
│   ├── MMHID-train/
│   │   ├── MMHID-train-DG-imgs.npy      # training images
│   │   └── MMHID-train-DG-labels.npy    # training labels
│   │   ├── MMHID-train-G-imgs.npy      # training images
│   │   └── MMHID-train-G-labels.npy    # training labels
│   └── MMHID-test/
│       ├── MMHID-test-DG-imgs.npy       # test images
│       └── MMHID-test-DG-labels.npy     # test labels
│       ├── MMHID-test-G-imgs.npy       # test images
│       └── MMHID-test-G-labels.npy     # test labels
├── ModelWeights/                   # model weight save directory
├── segmentation_models/            # segmentation-models source
├── Models for DG/
│   └── DG_Unet1_none.ipynb             # U‑Net (no pretrained backbone)
│   └── DG_Unet2_vgg.ipynb              # U‑Net (VGG16 backbone)
│   └── DG_Unet3_resnet34.ipynb         # U‑Net (ResNet34 backbone) ★
│   └── DG_Unet4_efficientnetb0.ipynb   # U‑Net (EfficientNetB0 backbone)
├── Models for G/
│   └── G_Unet1_none.ipynb              # U‑Net (no pretrained backbone)
│   └── G_Unet2_vgg.ipynb               # U‑Net (VGG16 backbone)
│   └── G_Unet3_resnet34.ipynb          # U‑Net (ResNet34)★
│   └── G_Unet4_efficientnetb0.ipynb    # U‑Net (EfficientNetB0)

**Note**: The `DG_` prefix indicates the digestive gland dataset, and the `G_` prefix indicates the gill filament dataset. This project is mainly implemented based on `DG_Unet3_resnet34.ipynb` and `G_Unet3_resnet34.ipynb`.

## 4. Dataset Description
### 4.1 Data Source
The dataset is stored in the `MMHID/` directory, containing both training and test sets. The data consist of digital images of pathological sections of mussel gill tissues and their corresponding binary label masks.
### 4.2 Data Format
File	Format	Description
`*-imgs.npy`	NumPy array	Original pathological images, shape `(N, 512, 512, 3)`
`*-labels.npy`	NumPy array	Binary annotation masks, shape `(N, 512, 512, 1)`
### 4.3  Data Loading

In [ ]:
import numpy as np

# Load training data
x_train = np.load('./MMHID/MMHID-train/MMHID-train-DG-imgs.npy')
y_train = np.load('./MMHID/MMHID-train/MMHID-train-DG-labels.npy')

# Load test data
x_test = np.load('./MMHID/MMHID-test/MMHID-test-DG-imgs.npy')
y_test = np.load('./MMHID/MMHID-test/MMHID-test-DG-labels.npy')

## 5. Model Architecture
### 5.1 Model Choice
We use the U‑Net implementation from `segmentation_models` with ResNet34 as the encoder backbone.

In [ ]:
from segmentation_models import Unet

model = Unet(
    backbone_name='resnet34',
    encoder_weights=None,          # no ImageNet pretraining
    input_shape=(512, 512, 3)
)

### 5.2 Network Structure
Total parameters: 24,456,154 – trainable: 24,438,804, non‑trainable: 17,350.

### Encoder Stages
| Stage             | Output Shape | Description               |
| ----------------- | ------------ | ------------------------- |
| Input             | 512×512×3    | RGB pathological image    |
| Conv0 + BN + ReLU | 256×256×64   | Initial convolution block |
| Pooling0          | 128×128×64   | Max pooling down‑sampling |
| Stage1            | 128×128×64   | 3 residual units          |
| Stage2            | 64×64×128    | 4 residual units          |
| Stage3            | 32×32×256    | 6 residual units          |
| Stage4            | 16×16×512    | 3 residual units          |

### Decoder Stages


| Stage          | Output Shape | Description                  |
| -------------- | ------------ | ---------------------------- |
| Decoder Stage0 | 32×32×256    | Upsampling + skip connection |
| Decoder Stage1 | 64×64×128    | Upsampling + skip connection |
| Decoder Stage2 | 128×128×64   | Upsampling + skip connection |
| Decoder Stage3 | 256×256×32   | Upsampling + skip connection |
| Decoder Stage4 | 512×512×16   | Upsampling                   |
| Final Conv     | 512×512×1    | Final convolution            |
| Sigmoid        | 512×512×1    | Binary activation            |

### 5.3 Image Data Format

In [ ]:
keras.backend.set_image_data_format('channels_last')
# The model uses channels_last, i.e., input shape (batch_size, height, width, channels).

## 6. Training Configuration
### 6.1 Model Compilation

In [ ]:
from tensorflow.keras.optimizers import Adam

def lr_schedule(epoch):
    return 1e-3  # fixed learning rate

model.compile(
    optimizer=Adam(learning_rate=lr_schedule(0)),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### 6.2 Training Parameters


| Parameter        | Value                | Description                              |
| ---------------- | -------------------- | ---------------------------------------- |
| Batch Size       | 50                   | samples per batch                        |
| Epochs           | 100                  | number of training epochs                |
| Validation Split | 0.4                  | 40% of training data used for validation |
| Optimizer        | Adam                 | Adaptive Moment Estimation               |
| Loss Function    | Binary Cross‑Entropy | for binary classification                |
| Learning Rate    | 1×10⁻³               | initial learning rate                    |

### 6.3 Callbacks

In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    LearningRateScheduler,
    ReduceLROnPlateau,
    TensorBoard
)

# TensorBoard logging
tensorboard = keras.callbacks.TensorBoard(logdir)

# Model checkpoint: save best weights based on validation loss
checkpoint = ModelCheckpoint(
    filepath=output_model_file,
    monitor='loss',
    verbose=1,
    save_best_only=True
)

# Learning rate scheduler
lr_scheduler = LearningRateScheduler(lr_schedule)

# Reduce learning rate on plateau
lr_reducer = ReduceLROnPlateau(
    factor=np.sqrt(0.1),  # ≈0.316
    cooldown=0,
    patience=5,
    min_lr=0.5e-6
)

callbacks = [tensorboard, checkpoint, lr_reducer, lr_scheduler]

### 6.4 Model Training

In [ ]:
history = model.fit(
    x=x_train,
    y=y_train,
    batch_size=50,
    epochs=100,
    validation_split=0.4,
    callbacks=callbacks
)

### 6.5 Model Weight Storage
During training, the best weights are saved to:
    `ModelWeights/DG_Unet3_resnet34_100epochs.hdf5 `  
After training, weights can be loaded as follows:

In [ ]:
model.load_weights('./ModelWeights/DG_Unet3_resnet34_100epochs.hdf5')

## 7. Model Evaluation
### 7.1 Prediction and Evaluation

In [ ]:
# Test set prediction
test = model.predict(x_test, batch_size=1, verbose=1)

# Evaluate on test set
model.evaluate(x_test, y_test)

### 7.2 Evaluation Metrics
We use the `medpy` library to compute common medical image segmentation metrics:

Metric    Function    Description

Dice Coefficient    `metric.binary.dc(pred, gt)`    segmentation similarity, [0,1]

Jaccard Index    `metric.binary.jc(pred, gt)`    Intersection over Union (IoU)

95% Hausdorff Distance    `metric.binary.hd95(pred, gt)`    boundary distance measure

Average Surface Distance    `metric.binary.asd(pred, gt)`    mean surface distance

In [ ]:
from medpy import metric

def calculate_metric_percase(pred, gt):
    dice = metric.binary.dc(pred, gt)
    jc = metric.binary.jc(pred, gt)
    hd = metric.binary.hd95(pred, gt)
    asd = metric.binary.asd(pred, gt)
    return dice, jc, hd, asd

### 7.3 Post‑processing (Binarization)
The model output probability map must be thresholded to obtain the final binary mask:

In [ ]:
# Threshold at 0.5
test[test > 0.5] = 1
test[test < 0.5] = 0

# Re‑compute metrics
calculate_metric_percase(test, y_test)

### 7.4 Training Curves Visualization

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

### 7.5 Prediction Visualization

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 15))
axs[0].imshow(x_test[i], cmap='gray')
axs[1].imshow(y_test[i], cmap='gray')
axs[2].imshow(test[i], cmap='gray')
axs[0].set_title("Image", fontsize=14)
axs[1].set_title("Label", fontsize=14)
axs[2].set_title("Predict", fontsize=14)
plt.grid(False)

## 8. Experimental Comparison
This project includes multiple backbone experiments for comparison:

| Notebook                        | Backbone       |
| ------------------------------- | -------------- |
| `DG_Unet1_none.ipynb`           | None           |
| `DG_Unet2_vgg.ipynb`            | VGG16          |
| `DG_Unet3_resnet34.ipynb`       | ResNet34       |
| `DG_Unet4_efficientnetb0.ipynb` | EfficientNetB0 |

## 9. Usage Instructions
### 9.1 Training the Model
1.Ensure the dataset is placed in the proper directory structure.

2.Open `DG_Unet3_resnet34.ipynb.`

3.Execute all cells sequentially.

4.After training, the best model weights are saved to `ModelWeights/.`

### 9.2 Using a Trained Model for Prediction

In [ ]:
# Load weights
model.load_weights('./ModelWeights/DG_Unet3_resnet34_100epochs.hdf5')

# Predict on new images
predictions = model.predict(new_images, batch_size=1, verbose=1)

# Binarize
predictions[predictions > 0.5] = 1
predictions[predictions < 0.5] = 0

### 9.3 Resizing Predictions to Original Resolution
If you need to resize the segmentation mask back to the original image size:

In [ ]:
import tensorflow as tf

resized = tf.compat.v1.image.resize_images(predictions, (3000, 4000))

## 10. Important Notes
1.GPU Memory: The model has ~24.5 million parameters. Ensure sufficient GPU VRAM (≥8 GB). If memory is limited, reduce `batch_size`.

2.Data Format: Input images must be RGB (3 channels) in `channels_last` format with size 512×512.

3.Weight Files: The best weights are saved as `DG_Unet3_resnet34_100epochs.hdf5` under `ModelWeights/`.

4.Reproducibility: For reproducible experiments, set a random seed before training.

5.TensorBoard Monitoring: Monitor training progress with:

In [ ]:
tensorboard --logdir=ModelWeights/